In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch2. Ollama - LLM 활용의 기본 개념 (LangChain)</font>

# 1. LLM을 활용한 답변 생성

## 1.1 Ollama를 통한 로컬 LLM 활용
- 개념 설명을 위한 오픈소스 모델 사용

**(internet) Ollama.com → (cmd) model pull**
: *ollama pull deepseek-r1:1.5b*

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')
result = llm.invoke('What is the capital of France?')
result # AI Message

In [ ]:
# response_metadata : 모델 실행에 대한 상세 정보 (전체 소요 시간, 모델 로딩 시간, 처리 토큰 수량 등)
result.response_metadata

In [ ]:
# content : 답변
print(result.content)

### Ollama에서 다운로드 및 실행 가능한 경량 모델
- ollama 모델 : 공식적인 한글 지원 x
    - ollama pull llama3.2:1b

- exaone 모델 : 공식적인 한글 지원 o
    - ollama pull exaone3.5:2.4b

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model='llama3.2:1b',
    temperature=0.2, # 창의성/무작위성 제어 (0.0에 가까울수록 정답 지향적·일관됨, 1.0에 가까울수록 무작위·창의적)
    top_k=40, # 다음 단어 생성 시 고려할 상위 후보 단어의 개수
    top_p=0.9, # 누적 확률 기반 단어 선택 범위 (Nucleus Sampling, 높을수록 다양한 표현 사용)
    num_ctx=4096, # 모델이 한 번에 처리 및 기억할 수 있는 최대 토큰 수 (컨텍스트 윈도우 크기)
)
result = llm.invoke('what is the capital of Korea?')
result.content

In [ ]:
result = llm.invoke('미국의 수도는 어디인가요?')
result.content

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model='exaone3.5:2.4b',
    temperature=0.2,
    top_k=40,
    top_p=0.9,
    num_ctx=4096,
)
result = llm.invoke('부산은 어떤 곳이야? 100자 내로 정리해줘')
print(result.content)

## 1.2 OpenAI 모델 활용

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')
result = llm.invoke('What is the capital of Korea?')
# result = llm.invoke('한국의 수도가 어디에요?')
result.content

In [ ]:
result

## 1.3 Claude 모델 활용

In [ ]:
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model='claude-haiku-4-5-20251001')
# llm.invoke('What is the capital of South Korea?')

# 2. LangChain 형식의 프롬프트 작성
- 프롬프트 : LLM에 입력으로 전달되는 텍스트 및 메시지 구조

## 2.1 기본 프롬프트 템플릿
- PromptTemplate : 변수가 포함된 템플릿을 정의하고, 변수값을 주입하여 PromptValue 생성
- PromptValue :  완성된 프롬프트 데이터를 string 또는 messages 중 원하는 형태로 변환할 수 있도록 감싸둔 표준 전달 객체

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
result = llm.invoke('What is the capital of France?')

In [ ]:
# 프롬프트 가능 타입 : str, PromptValue, list of BaseMessages
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
    template="What is the capital of {country}?", # {변수명} 형태로 동적 값 지정
    input_variables = ['country'] 
)
prompt = prompt_template.invoke({'country':'Korea'})
print(1, type(prompt))
prompt = prompt_template.invoke('Korea')
print(2, prompt)
llm.invoke(prompt)

In [ ]:
def answer(country):
    '나라명을 입력 받은 후 llm 모델을 거쳐 수도명을 받아 return '
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    llm = ChatOllama(model='llama3.2:1b')
    prompt_template = PromptTemplate(
        template='What is the capital of {country}',
        input_variables = ['country']
    )
    result = llm.invoke(prompt_template.invoke(country))
    return result.content

In [ ]:
country = input('수도가 궁금한 나라의 영어명 :')
answer(country)

## 2.2 메시지 기반 프롬프트 (Message-based Prompt)
- **입력 타입**: List[BaseMessage] -> `[BaseMessage 객체 1, BaseMessage 객체 2, ...]`

### BaseMessage 상속 주요 클래스
1. SystemMessage : AI의 역할 및 제약 조건 설정
2. HumanMessage : 사용자의 입력 데이터/질문
3. AIMessage : 모델의 이전 응답 기록
4. ToolMessage : 에이전트의 도구 실행 결과를 모델에 전달

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="'You are a helpful assistant!"), # AI의 역할 (페르소나)
    HumanMessage(content="What is the capital of Italy?"), # 질문 예시  
    AIMessage(content='The capital of Italy is Rome.'), # 답변 예시
    HumanMessage(content="What is the capital of Italy?"), # 질문-답변 예시가 2개 이상 : Few-Shot 모델 (Zero-Shot, One-Shot)
    AIMessage(content='The capital of Italy is Rome.'), 
    HumanMessage(content='What is the captital of Korea?') # 이전 맥락(AIMessage)을 기억한 상태에서 질문 
]
llm.invoke(message_list)

## 2.3 ChatPromptTemplate
- 대화형 모델(Chat Model)에 적합한 메시지 리스트 구조의 템플릿 생성 클래스
- BaseMessage 객체 대신 `(role, template)` 형태의 튜플 리스트로 작성

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    ("system","'You are a helpful assistant!"), # = SystemMessage(content="'You are a helpful assistant!")
    ("human","What is the capital of Italy?"), # = HumanMessage(content="What is the capital of Italy?"), 
    ('ai','The capital of Italy is Rome.'), # = AIMessage(content='The capital of Italy is Rome.'), 
    ("human","What is the capital of Italy?"),
    ('ai','The capital of Italy is Rome.'),
    ("human",'What is the captital of {country}?')
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :',prompt)

In [ ]:
llm.invoke(chatPromptTemplate.invoke('Korea'))

# 3. 답변 형식 조정
- LLM의 실행 결과인 AIMessage 객체를 사용자 및 시스템이 활용하기 쉬운 타입(`str`, `json`, `pydantic` 등)으로 파싱/변환하는 도구

## 3.1 StrOutputParser 활용
- AIMessage에서 텍스트 데이터만 추출하여 단순 문자열 형태로 변환

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# 명시적인 지시사항이 포함된 프롬프트 템플릿 작성
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)

# 프롬프트 템플릿에 값 주입 (dict 형태로 전달)
prompt = prompt_template.invoke({'country': 'Korea'})
print('프롬프트 :', prompt)

# LLM에 질문 (PromptValue 객체를 직접 입력)
aimessage = llm.invoke(prompt)

# AIMessage에서 순수 문자열(str)로 변환
output_parser = StrOutputParser()
result = output_parser.invoke(aimessage)
print('결과 :', result)

In [ ]:
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)

output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))

In [ ]:
chatPromptTemplate = ChatPromptTemplate([
    ("system","'You are a helpful assistant!"),
    ("human","What is the capital of Italy?"),
    ('ai','The capital of Italy is Rome.'),
    ("human","What is the capital of Italy?"),
    ('ai','The capital of Italy is Rome.'),
    ("human",'What is the capital of {country}? Return the name of the city only.')
])
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(chatPromptTemplate.invoke({'country':'Korea'})))

## 3.2 JsonOutputParser 활용
- LLM이 생성한 텍스트 형태의 응답을 JSON 객체로 파싱/변환하는 도구

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

llm = ChatOllama(model='llama3.2:1b')

country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}.
        - Capital
        - Population
        - Language
        - Currency
    Return ONLY a valid JSON object with no additional text.
    Example format :
    {{"Capital":"Seoul", "Population":"51,000,000", "Language":"Korean", "Currency":"Won"}}""",
    input_variables = ['country']
)

prompt = country_detail_prompt.invoke({'country':'France'})
aimessage = llm.invoke(prompt)
output_parser = JsonOutputParser()
result = output_parser.invoke(aimessage)
print(type(result), result)

In [ ]:
output_parser.invoke(llm.invoke(country_detail_prompt.invoke('France')))

## 3.3 Pydantic
- 데이터 검증 및 설정 관리 라이브러리
- 입력받은 데이터의 타입을 자동으로 검증하고 변환 : Python 객체, Dict, JSON 등

In [ ]:
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
user = User('1', '홍길동')
print(user)
print(user.id, user.name, user.is_active)

In [ ]:
from pydantic import BaseModel, Field
class User(BaseModel):
    # gt=0 : id>0, lt=0 : id<0, ge=0 : id>=0, le=0 : id<=0
    id:int         = Field(gt=0, description='id')
    name:str       = Field(min_length=2, description='name')
    is_active:bool = Field(default=True, description='ID 활성화')
user = User(id=1, name='홍길동')
print(user)

In [ ]:
# Dict 형식 출력 프롬프트
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}.
        - Capital
        - Population
        - Language
        - Currency
    Return ONLY a valid JSON object with no additional text.""",
    input_variables = ['country']
)

# Pydantic 스키마 정의
class CountryDetail(BaseModel):
    capital:str    = Field(description='the capital of the country')
    population:int = Field(description='the population of the country')
    language:str   = Field(description='the language of the country')
    currency:str   = Field(description='the currency of the country')
        
# 구조화된 출력을 지원하는 LLM 객체 생성 
# pydantic의 BaseModel을 상속 받은 클래스만 가능
structedllm = llm.with_structured_output(CountryDetail)
info = structedllm.invoke(country_detail_prompt.invoke({'country':'Korea'}))
print(type(info))
print(info.capital, info.population, info.language, info.currency)
print(info.model_dump()) # 객체 → Dict 변환

In [ ]:
aimessage = llm.invoke(country_detail_prompt.invoke({'country':'Korea'}))
print(aimessage.content)

# 4. LCEL
- LangChain Expression Language을 활용한 langchain 생성

## 4.1 StrOutputParser 활용

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)

output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))

# 4.2 LCEL 활용

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)

output_parser = StrOutputParser()

# LCEL (파이프라인) 방식 - 체인 구성: Prompt → LLM → Parser
chain = prompt_template | llm | output_parser
info = chain.invoke({'country': 'Korea'})
info

## 4.3 복합 체인 구성
- 여러 단계의 추론이 필요한 경우

In [ ]:
# 국가 설명 → 국가명
country_prompt = PromptTemplate(
    template="""
    Guess the name of the country based on the following information:
    {information}
    Return the name of the country only
    """,
    input_variables=['information']
)

outputParser = StrOutputParser()

country_chain = country_prompt | llm | outputParser
country = country_chain.invoke({'information':'This country is very famous for its wine'})
print(country)

capital_chain = prompt_template | llm | outputParser
print(capital_chain.invoke({'country': country}))

In [ ]:
# 복합 체인 방법 1
final_chain = country_chain | capital_chain # 국가 설명 → 국가명 → 국가 수도 
final_chain.invoke({'information':'This country is very famous for its wine'})

In [ ]:
# 복합 체인 방법 2
final_chain = {'country':country_chain} | capital_chain
final_chain.invoke({'information':'This country is very famous for BTS'})

```
프롬프트(PromptTemplate) : 페르소나 설정, 예시(Few-shot), 변수를 결합해 LLM에 전달할 질문 조립
LLM(ChatOpenAI, ChatOllama) : 클라우드 API나 로컬 모델을 통일된 방식으로 호출해 답변 생성
출력 파서(with_structured_output / Pydantic) : LLM의 줄글 응답을 코드에서 JSON이나 Pydantic 객체로 정제
체인 및 RAG 확장 : 각 단계의 invoke() 결과를 파이프라인으로 연결하여 문서 검색(RAG)이나 복합 추론 기능 구현
```